# Restauración fotográfica con difusión en Google Colab

Este notebook ejecuta la aplicación Gradio completa en Colab. Antes de continuar, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, elegí una GPU. El notebook verifica que CUDA esté realmente disponible.

Prepará un ZIP del proyecto que contenga `app.py`, `restaurador/` y `pyproject.toml`. No incluyas `.venv/`, `__pycache__/` ni `.pytest_cache/`. Desde PowerShell, por ejemplo:

```powershell
Compress-Archive -Path app.py,restaurador\*.py,pyproject.toml -DestinationPath restauracion-colab.zip
```

El enlace público de Gradio permite acceder a la interfaz mientras el runtime esté activo. Compartilo solo con personas de confianza y no cargues fotografías sensibles.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

subidos = files.upload()
archivos_zip = [nombre for nombre in subidos if nombre.lower().endswith(".zip")]
if len(archivos_zip) != 1:
    raise ValueError("Subí exactamente un archivo ZIP del proyecto.")

carpeta_destino = Path("/content/restauracion_mvp").resolve()
carpeta_destino.mkdir(parents=True, exist_ok=True)
with ZipFile(archivos_zip[0]) as archivo_zip:
    for miembro in archivo_zip.infolist():
        ruta_destino = (carpeta_destino / miembro.filename).resolve()
        if not ruta_destino.is_relative_to(carpeta_destino):
            raise ValueError("El ZIP contiene una ruta de extracción no válida.")
    archivo_zip.extractall(carpeta_destino)

archivos_proyecto = list(carpeta_destino.rglob("pyproject.toml"))
if len(archivos_proyecto) != 1:
    raise FileNotFoundError("El ZIP debe contener un único pyproject.toml.")
PROJECT_DIR = archivos_proyecto[0].parent
%cd $PROJECT_DIR

In [ ]:
# Colab ya incluye PyTorch compilado para el runtime; no reinstalar torch aquí.
%pip install "accelerate>=1.1,<2" "diffusers>=0.34,<1" "gradio>=5,<6" "numpy>=1.26,<3" "pillow>=10,<13" "transformers>=4.45,<5"

In [ ]:
import sys
import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"El proyecto requiere Python 3.12; Colab está usando {sys.version.split()[0]}."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible. Seleccioná una GPU en el tipo de entorno de ejecución de Colab y volvé a ejecutar."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")

## Iniciar la aplicación

La celda siguiente queda activa mientras se usa la app. Abrí el enlace `gradio.live` que aparece en la salida. La primera ejecución descarga los modelos; después, el proceso los conserva en caché en la GPU.

In [ ]:
import app

app.demo.queue().launch(
    server_name="0.0.0.0",
    share=True,
    inbrowser=False,
    show_error=True,
)